# Imports

In [ ]:
import pathlib
import tqdm
import pandas as pd
import numpy as np
import torch

from experiments.models import (
    HyperTreeARForecast,
    HyperTreeNetARForecast,
    HyperTreeETSForecast,
)

from experiments.utils import load_experiments_specs

# Load Experiment Specifications

In [ ]:
dataset = "ABC"

In [ ]:
# Load specifications
train_type = "local"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

# Write AR parameters only for datasets the figure notebooks read
# (time_varying_params.ipynb).
save_extras = dataset.lower() == "airpassengers"

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]
data_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

# Meta Data
meta = experiment_specs["meta"]
features = meta["features"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]
max_lag = meta["max_lag"]
series_ids = meta["series_ids"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 123

# Hyper-Parameters
config = experiment_specs["config"]
loss_fn = config["general"]["loss_function"]

dl_params = config["deep_learning"]

htar_params = config["hyper_tree_params"]
htets_params = config["hyper_tree_ets_params"]
seasonality_ets = "month_feat_ts"

htnetar_params = config["hyper_treenet_params"]
htnetar_params_lgb = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "embedding_dimension", "use_random_projection"]}
htnetar_network_params = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "eta", "linear_tree"]}
htnetar_network_params["rp_embed_dim"] = max_lag
htnetar_network_params["hidden_dim"] = dl_params["hidden_size"]
htnetar_network_params["dropout"] = dl_params["dropout"]
htnetar_network_params["learning_rate"] = dl_params["learning_rate"]

# Run Models

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

model_forecasts = []
extra_params = []
extra_embeddings = []

for series in tqdm.tqdm(series_ids, total=len(series_ids)):

    ###
    # Subset Data
    ###
    series_df = data_df[data_df["series_id"] == series].reset_index(drop=True).copy()

    ###
    # Split Data into Train-Test
    ###
    test = series_df.tail(fcst_h)
    train = series_df.drop(test.index)

    ###
    # Hyper-Tree-AR
    ###
    if save_extras:
        ht_ar_fcst, ht_ar_params = HyperTreeARForecast(
            htar_params,
            train,
            test.drop(columns=["value"]),
            features,
            freq,
            fcst_h,
            max_lag,
            loss_fn,
            seed,
            return_extras=True,
        )
        extra_params.append(ht_ar_params)
    else:
        ht_ar_fcst = HyperTreeARForecast(
            htar_params,
            train,
            test.drop(columns=["value"]),
            features,
            freq,
            fcst_h,
            max_lag,
            loss_fn,
            seed,
        )

    ###
    # Hyper-TreeNet-AR
    ###
    if save_extras:
        htnet_ar_fcst, htnet_ar_params, htnet_ar_embeddings = HyperTreeNetARForecast(
            htnetar_params,
            htnetar_params_lgb,
            htnetar_network_params,
            train,
            test.drop(columns=["value"]),
            features,
            freq,
            fcst_h,
            max_lag,
            loss_fn,
            seed,
            device,
            return_extras=True,
        )
        extra_params.append(htnet_ar_params)
        extra_embeddings.append(htnet_ar_embeddings)
    else:
        htnet_ar_fcst = HyperTreeNetARForecast(
            htnetar_params,
            htnetar_params_lgb,
            htnetar_network_params,
            train,
            test.drop(columns=["value"]),
            features,
            freq,
            fcst_h,
            max_lag,
            loss_fn,
            seed,
            device,
        )

    ###
    # Hyper-Tree-ETS
    ###
    ht_ets_fcst = HyperTreeETSForecast(
        htets_params,
        train,
        test.drop(columns=["value"]),
        features,
        freq,
        fcst_h,
        seasonality_ets,
        loss_fn,
        manual_param=htets_params["manual_param"],
    )
    ###
    # Append Forecasts
    ###
    model_forecasts.append(
        pd.concat(
            [
                ht_ar_fcst,
                htnet_ar_fcst,
                ht_ets_fcst,
            ],
            axis=0
        )
    )

# Combine Forecasts

In [ ]:
# Concatenate all forecasts into a single DataFrame
fcsts_df = pd.concat([
    pd.concat(model_forecasts, axis=0),
], axis=0).reset_index(drop=True)

# Add actual values and dataset information
fcsts_df = pd.merge(
    fcsts_df,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)

# Save Forecasts and (optionally) AR parameters

In [ ]:
repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / train_type
result_path.mkdir(parents=True, exist_ok=True)

fcsts_df.to_csv(result_path / f"{dataset.lower()}_hypertrees_fcsts.csv", index=False)

if save_extras:
    params_df = pd.concat(extra_params, axis=0, ignore_index=True)
    params_df.to_csv(result_path / f"{dataset.lower()}_ar_parameters.csv", index=False)

    embeddings_df = pd.concat(extra_embeddings, axis=0, ignore_index=True)
    embeddings_df.to_csv(result_path / f"{dataset.lower()}_tree_embeddings.csv", index=False)